In [2]:
# ==========================================================
# CELL 1 : Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# ==========================================================
# Set Random Seed for Reproducibility
# ==========================================================

import random
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random Seed Fixed Successfully!")

Random Seed Fixed Successfully!


In [4]:
# ============================
# Load Preprocessed Dataset
# ============================

import joblib

X_train = joblib.load("Demand_X_train.pkl")
X_test = joblib.load("Demand_X_test.pkl")

y_train = joblib.load("Demand_y_train.pkl")
y_test = joblib.load("Demand_y_test.pkl")

print("Training Features Shape :", X_train.shape)
print("Testing Features Shape  :", X_test.shape)

print("\nTraining Target Shape :", y_train.shape)
print("Testing Target Shape  :", y_test.shape)

Training Features Shape : (100, 17)
Testing Features Shape  : (26, 17)

Training Target Shape : (100,)
Testing Target Shape  : (26,)


In [5]:
# ============================
# Feature Scaling
# ============================

from sklearn.preprocessing import MinMaxScaler

# Feature Scaler
X_scaler = MinMaxScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

# Target Scaler
y_scaler = MinMaxScaler()

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

print("Feature Scaling Completed Successfully!")
print("X_train_scaled :", X_train_scaled.shape)
print("X_test_scaled  :", X_test_scaled.shape)

print("y_train_scaled :", y_train_scaled.shape)
print("y_test_scaled  :", y_test_scaled.shape)

Feature Scaling Completed Successfully!
X_train_scaled : (100, 17)
X_test_scaled  : (26, 17)
y_train_scaled : (100, 1)
y_test_scaled  : (26, 1)


In [6]:
# ============================
# Reshape Data for LSTM
# ============================

X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    1,
    X_train_scaled.shape[1]
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    1,
    X_test_scaled.shape[1]
)

print("Training Shape :", X_train_lstm.shape)
print("Testing Shape  :", X_test_lstm.shape)

Training Shape : (100, 1, 17)
Testing Shape  : (26, 1, 17)


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.layers import GRU
from tensorflow.keras.callbacks import EarlyStopping

In [8]:
# ============================
# Early Stopping
# ============================

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

print("EarlyStopping Created Successfully!")

EarlyStopping Created Successfully!


In [9]:
# ============================
# Build Bidirectional LSTM Model
# ============================

model = Sequential()

# Bidirectional LSTM Layer
model.add(
    Bidirectional(
        LSTM(
            units=64,
            activation='tanh'
        ),
        input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])
    )
)

# Dropout Layer
model.add(Dropout(0.3))

# Output Layer
model.add(Dense(1))

# Compile Model
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

model.compile(
    optimizer=optimizer,
    loss='mse'
)

print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        41,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,113 (164.50 KB)

 Trainable params: 42,113 (164.50 KB)

 Non-trainable params: 0 (0.00 B)

None


In [10]:
history = model.fit(
    X_train_lstm,
    y_train_scaled,
    epochs=200,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - loss: 0.1493 - val_loss: 0.1358
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0311 - val_loss: 0.0242
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0277 - val_loss: 0.0203
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0205 - val_loss: 0.0253
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0172 - val_loss: 0.0302
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0171 - val_loss: 0.0197
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0122 - val_loss: 0.0145
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0131 - val_loss: 0.0175
Epoch 9/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0105 - val_loss: 0.0163
Epoch 10/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0097 - val_loss: 0.0121
Epoch 11/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0120 - val_loss: 0.0142
Epoch 12/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0

In [11]:
# ============================
# Make Predictions
# ============================

# Training Predictions
train_predictions = model.predict(X_train_lstm)

# Testing Predictions
test_predictions = model.predict(X_test_lstm)

# Convert predictions back to original scale
train_predictions = y_scaler.inverse_transform(train_predictions)
test_predictions = y_scaler.inverse_transform(test_predictions)

# Convert actual values back to original scale
y_train_actual = y_scaler.inverse_transform(y_train_scaled)
y_test_actual = y_scaler.inverse_transform(y_test_scaled)

print("Prediction Completed Successfully!")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Prediction Completed Successfully!


In [12]:
# ============================
# Model Evaluation
# ============================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import numpy as np

# -----------------------------
# Training Metrics
# -----------------------------
train_mae = mean_absolute_error(y_train_actual, train_predictions)
train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_predictions))
train_mape = mean_absolute_percentage_error(y_train_actual, train_predictions) * 100
train_r2 = r2_score(y_train_actual, train_predictions)

print("=" * 50)
print("Training Performance")
print("=" * 50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape:.2f}%")
print(f"R²   : {train_r2:.4f}")

# -----------------------------
# Testing Metrics
# -----------------------------
test_mae = mean_absolute_error(y_test_actual, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test_actual, test_predictions))
test_mape = mean_absolute_percentage_error(y_test_actual, test_predictions) * 100
test_r2 = r2_score(y_test_actual, test_predictions)

print("\n")
print("=" * 50)
print("Testing Performance")
print("=" * 50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 201.20
RMSE : 274.33
MAPE : 2.23%
R²   : 0.9144


Testing Performance
MAE  : 299.00
RMSE : 398.63
MAPE : 2.86%
R²   : 0.8356
